#Realización de cubo y sabana

## 1. Imports y carga de archivos limpios


In [ ]:
import os

import numpy as np
import pandas as pd

# Rutas del paso 02.
# Entrada: CSV limpios generados por el script 01.
# Salida: sabanas en csv/ y cubos/dimensiones/hechos en csv/cubos/.
CSV_LIMPIOS_DIR = os.environ.get('CSV_LIMPIOS_DIR', '../csv/csv_limpios')
SABANAS_DIR = os.environ.get('SABANAS_DIR', '../csv')
CUBOS_DIR = os.environ.get('CUBOS_DIR', '../csv/cubos')

CSV_LIMPIOS_DIR = os.path.normpath(CSV_LIMPIOS_DIR)
SABANAS_DIR = os.path.normpath(SABANAS_DIR)
CUBOS_DIR = os.path.normpath(CUBOS_DIR)

os.makedirs(SABANAS_DIR, exist_ok=True)
os.makedirs(CUBOS_DIR, exist_ok=True)

SUBIR_A_BIGQUERY = False

PIPELINE_AUDIT = {
    'leidos': {},
    'generados': {},
}


def path_limpio(nombre):
    return os.path.join(CSV_LIMPIOS_DIR, nombre)


def path_sabana(nombre):
    return os.path.join(SABANAS_DIR, nombre)


def path_cubo(nombre):
    return os.path.join(CUBOS_DIR, nombre)


def resumen_dataframe(df):
    return {
        'filas': len(df),
        'columnas': list(df.columns),
        'tipos_principales': {col: str(dtype) for col, dtype in df.dtypes.head(20).items()},
        'nulos_principales': df.isna().sum().sort_values(ascending=False).head(10).to_dict(),
        'duplicados_exactos': int(df.duplicated().sum()),
    }


def leer_csv_limpio(nombre, **kwargs):
    ruta = path_limpio(nombre)
    df = pd.read_csv(ruta, **kwargs)
    info = resumen_dataframe(df)
    info['ruta'] = ruta
    PIPELINE_AUDIT['leidos'][nombre] = info
    return df


def guardar_sabana(df, nombre):
    ruta = path_sabana(nombre)
    df.to_csv(ruta, index=False)
    info = resumen_dataframe(df)
    info['ruta'] = ruta
    info['tipo_salida'] = 'sabana'
    PIPELINE_AUDIT['generados'][nombre] = info
    return ruta


def guardar_cubo(df, nombre):
    ruta = path_cubo(nombre)
    df.to_csv(ruta, index=False)
    info = resumen_dataframe(df)
    info['ruta'] = ruta
    info['tipo_salida'] = 'cubo'
    PIPELINE_AUDIT['generados'][nombre] = info
    return ruta

# Cargamos los archivos limpios
orders = leer_csv_limpio('olist_orders_clean.csv')
items = leer_csv_limpio('olist_order_items_clean.csv')
payments = leer_csv_limpio('olist_order_payments_dataset_clean.csv')
reviews = leer_csv_limpio('olist_order_reviews_dataset_clean.csv')
products = leer_csv_limpio('olist_products_clean.csv')
customers = leer_csv_limpio('olist_customers_clean.csv')
geolocation = leer_csv_limpio('olist_geolocation_clean.csv')
sellers = leer_csv_limpio('olist_sellers_dataset_clean.csv')
category_translation = leer_csv_limpio('product_category_name_translation_clean.csv')

print('Archivos cargados correctamente desde:', CSV_LIMPIOS_DIR)
print('Salida de sabanas:', SABANAS_DIR)
print('Salida de cubos  :', CUBOS_DIR)
for nombre, info in PIPELINE_AUDIT['leidos'].items():
    print(f"- {nombre}: {info['filas']:,} filas | {len(info['columnas'])} columnas")

### 2. Normalización y conversiones de tipos


In [ ]:
# Normalizar nombres de columnas de status
orders["order_status"] = orders["order_status"].astype(str).str.strip().str.lower()

# Convertir fechas
columnas_fecha_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in columnas_fecha_orders:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"], errors="coerce")
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"], errors="coerce")
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], errors="coerce")

# Normalizar ciudades a minúsculas para consistencia
customers["customer_city"] = customers["customer_city"].astype(str).str.strip().str.lower()
geolocation["geolocation_city"] = geolocation["geolocation_city"].astype(str).str.strip().str.lower()
sellers["seller_city"] = sellers["seller_city"].astype(str).str.strip().str.lower()




### 3. Feature engineering en `orders`


In [ ]:
#Feature engineering en orders
orders["fecha"] = orders["order_purchase_timestamp"].dt.date
orders["anio"] = orders["order_purchase_timestamp"].dt.year
orders["mes"] = orders["order_purchase_timestamp"].dt.month
orders["dia"] = orders["order_purchase_timestamp"].dt.day
orders["trimestre"] = orders["order_purchase_timestamp"].dt.quarter
orders["dia_semana_num"] = orders["order_purchase_timestamp"].dt.dayofweek
orders["dia_semana_nombre"] = orders["order_purchase_timestamp"].dt.day_name()
orders["es_fin_semana"] = orders["dia_semana_num"].isin([5, 6]).astype(int)

# Métricas logísticas
orders["delivery_days_real"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["delivery_days_estimated"] = (
    orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

orders["delivery_delay_days"] = orders["delivery_delay_days"].fillna(0)
orders["is_late_delivery"] = (orders["delivery_delay_days"] > 0).astype(int)
orders["is_delivered"] = (orders["order_status"] == "delivered").astype(int)

# Clave de fecha
orders["fecha_key"] = orders["order_purchase_timestamp"].dt.strftime("%Y%m%d")




### 4. Feature engineering en `products` (traducción robusta de categorías)


In [ ]:
# Feature engineering en products
products['product_volume_cm3'] = (
    products['product_length_cm'].fillna(0) *
    products['product_height_cm'].fillna(0) *
    products['product_width_cm'].fillna(0)
)

# Unir traduccion de categorias.
# La limpieza deja `product_category_name` en Title Case ("Esporte Lazer")
# mientras el catalogo conserva snake_case ("esporte_lazer").
# Se conserva la columna original como display y se hace el lookup sobre una llave normalizada.
def _to_snake(name):
    if not isinstance(name, str):
        return name
    return name.strip().lower().replace(' ', '_')

products['_cat_key'] = products['product_category_name'].map(_to_snake)
category_translation = category_translation.copy()
category_translation['_cat_key'] = category_translation['product_category_name'].map(_to_snake)
products = products.merge(
    category_translation[['_cat_key', 'product_category_name_english']],
    on='_cat_key',
    how='left'
)

# Logica incorporada desde el script 03: aplicar mapeo canonico antes de generar
# dimensiones, hechos y sabanas. Solo cambia el contenido de product_category_name_english.
TRANSLATION = {
    'agro_industria_e_comercio': 'agro_industry_and_commerce',
    'alimentos': 'food',
    'alimentos_bebidas': 'food_drink',
    'artes': 'art',
    'artes_e_artesanato': 'arts_and_craftsmanship',
    'artigos_de_festas': 'party_supplies',
    'artigos_de_natal': 'christmas_supplies',
    'audio': 'audio',
    'automotivo': 'auto',
    'bebes': 'baby',
    'bebidas': 'drinks',
    'beleza_saude': 'health_beauty',
    'brinquedos': 'toys',
    'cama_mesa_banho': 'bed_bath_table',
    'casa_conforto': 'home_comfort',
    'casa_conforto_2': 'home_comfort_2',
    'casa_construcao': 'home_construction',
    'cds_dvds_musicais': 'cds_dvds_musicals',
    'cine_foto': 'cine_photo',
    'climatizacao': 'air_conditioning',
    'consoles_games': 'consoles_games',
    'construcao_ferramentas_construcao': 'construction_tools_construction',
    'construcao_ferramentas_ferramentas': 'construction_tools_tools',
    'construcao_ferramentas_iluminacao': 'construction_tools_lights',
    'construcao_ferramentas_jardim': 'construction_tools_garden',
    'construcao_ferramentas_seguranca': 'construction_tools_safety',
    'cool_stuff': 'cool_stuff',
    'dvds_blu_ray': 'dvds_blu_ray',
    'eletrodomesticos': 'home_appliances',
    'eletrodomesticos_2': 'home_appliances_2',
    'eletronicos': 'electronics',
    'eletroportateis': 'small_appliances',
    'esporte_lazer': 'sports_leisure',
    'fashion_bolsas_e_acessorios': 'fashion_bags_accessories',
    'fashion_calcados': 'fashion_shoes',
    'fashion_esporte': 'fashion_sport',
    'fashion_roupa_feminina': 'fashion_female_clothing',
    'fashion_roupa_infanto_juvenil': 'fashion_childrens_clothes',
    'fashion_roupa_masculina': 'fashion_male_clothing',
    'fashion_underwear_e_moda_praia': 'fashion_underwear_beach',
    'ferramentas_jardim': 'garden_tools',
    'flores': 'flowers',
    'fraldas_higiene': 'diapers_and_hygiene',
    'industria_comercio_e_negocios': 'industry_commerce_and_business',
    'informatica_acessorios': 'computers_accessories',
    'instrumentos_musicais': 'musical_instruments',
    'la_cuisine': 'la_cuisine',
    'livros_importados': 'books_imported',
    'livros_interesse_geral': 'books_general_interest',
    'livros_tecnicos': 'books_technical',
    'malas_acessorios': 'luggage_accessories',
    'market_place': 'market_place',
    'moveis_colchao_e_estofado': 'furniture_mattress_and_upholstery',
    'moveis_cozinha_area_de_servico_jantar_e_jardim': 'kitchen_dining_laundry_garden_furniture',
    'moveis_decoracao': 'furniture_decor',
    'moveis_escritorio': 'office_furniture',
    'moveis_quarto': 'furniture_bedroom',
    'moveis_sala': 'furniture_living_room',
    'musica': 'music',
    'papelaria': 'stationery',
    'pc_gamer': 'pc_gamer',
    'pcs': 'computers',
    'perfumaria': 'perfumery',
    'pet_shop': 'pet_shop',
    'portateis_casa_forno_e_cafe': 'small_appliances_home_oven_and_coffee',
    'portateis_cozinha_e_preparadores_de_alimentos': 'small_appliances_kitchen_food_preparation',
    'relogios_presentes': 'watches_gifts',
    'seguros_e_servicos': 'security_and_services',
    'sinalizacao_e_seguranca': 'signaling_and_security',
    'tablets_impressao_imagem': 'tablets_printing_image',
    'telefonia': 'telephony',
    'telefonia_fixa': 'fixed_telephony',
    'utilidades_domesticas': 'housewares',
    'sin_categoria': 'no_category',
}

products['product_category_name_english'] = products['_cat_key'].map(TRANSLATION)

sin_match = products[
    products['product_category_name_english'].isna() &
    products['_cat_key'].notna()
]['_cat_key'].value_counts()

print('Categorias sin match:', len(sin_match))
if len(sin_match) > 0:
    print(sin_match)
    products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')

products = products.drop(columns='_cat_key')

assert products['product_category_name_english'].notna().all() or products['product_category_name'].isna().all(), (
    'product_category_name_english quedo con nulos no atribuibles a categoria nula. '
    'Revisar el catalogo product_category_name_translation_clean.csv.'
)

consistencia = products.groupby('product_category_name')['product_category_name_english'].nunique()
assert (consistencia == 1).all(), 'Hay categorias PT con multiples traducciones EN'

print('Traduccion de categorias aplicada antes de generar sabanas y cubos.')
print('Nulos en product_category_name_english:', products['product_category_name_english'].isna().sum())

### 5. Agregaciones (geolocalización, pagos, reviews)


In [ ]:
#Parte de geolocalización
def moda_segura(serie):
    moda = serie.mode()
    if len(moda) > 0:
        return moda.iloc[0]
    return serie.iloc[0] if len(serie) > 0 else np.nan

geolocation_agg = (
    geolocation
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg({
        "geolocation_lat": "mean",
        "geolocation_lng": "mean",
        "geolocation_city": moda_segura,
        "geolocation_state": moda_segura
    })
    .rename(columns={
        "geolocation_zip_code_prefix": "zip_code_prefix",
        "geolocation_lat": "geo_lat",
        "geolocation_lng": "geo_lng",
        "geolocation_city": "geo_city",
        "geolocation_state": "geo_state"
    })
)

#Resumen de pagos por pedido
payments_agg = (
    payments
    .groupby("order_id", as_index=False)
    .agg({
        "payment_value": "sum",
        "payment_installments": "max"
    })
)

payment_type_mode = (
    payments.groupby("order_id")["payment_type"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
    .reset_index()
)

payments_agg = payments_agg.merge(payment_type_mode, on="order_id", how="left")

#Resumen de reseñas por pedido
reviews_agg = (
    reviews
    .groupby("order_id", as_index=False)
    .agg({
        "review_score": "mean",
        "flag_short_message": "max",
        "flag_answer_before_creation": "max",
        "flag_review_id_duplicated": "max",
        "flag_review_id_multi_order": "max"
    })
)

reviews_agg["review_score"] = reviews_agg["review_score"].round(2)



### 6. Dimensiones (con sentinela `SIN_PAGO`)


In [ ]:
#Dimensiones

dim_tiempo = (
    orders[[
        "fecha_key",
        "fecha",
        "anio",
        "mes",
        "dia",
        "trimestre",
        "dia_semana_num",
        "dia_semana_nombre",
        "es_fin_semana"
    ]]
    .drop_duplicates()
    .copy()
)

meses = {
    1: "enero", 2: "febrero", 3: "marzo", 4: "abril",
    5: "mayo", 6: "junio", 7: "julio", 8: "agosto",
    9: "septiembre", 10: "octubre", 11: "noviembre", 12: "diciembre"
}
dim_tiempo["nombre_mes"] = dim_tiempo["mes"].map(meses)

# Reordenar columnas
dim_tiempo = dim_tiempo[[
    "fecha_key", "fecha", "anio", "mes", "nombre_mes", "trimestre",
    "dia", "dia_semana_num", "dia_semana_nombre", "es_fin_semana"
]]


dim_cliente = customers.copy()


dim_geografia_cliente = (
    customers[["customer_zip_code_prefix"]]
    .drop_duplicates()
    .rename(columns={"customer_zip_code_prefix": "zip_code_prefix"})
    .merge(geolocation_agg, on="zip_code_prefix", how="left")
)


dim_producto = products.copy()

dim_vendedor = sellers.copy()

# Agregar geolocalización de vendedor
geo_seller = geolocation_agg.rename(columns={
    "zip_code_prefix": "seller_zip_code_prefix",
    "geo_lat": "seller_geo_lat",
    "geo_lng": "seller_geo_lng",
    "geo_city": "seller_geo_city",
    "geo_state": "seller_geo_state"
})

dim_vendedor = dim_vendedor.merge(
    geo_seller,
    on="seller_zip_code_prefix",
    how="left"
)


dim_pago = (
    payments[["payment_type", "payment_installments"]]
    .drop_duplicates()
    .sort_values(["payment_type", "payment_installments"])
    .reset_index(drop=True)
)

dim_pago["pago_key"] = range(1, len(dim_pago) + 1)

# Vincular pago_key a payments_agg
payments_agg = payments_agg.merge(
    dim_pago,
    on=["payment_type", "payment_installments"],
    how="left"
)

# Reordenar columnas
dim_pago = dim_pago[["pago_key", "payment_type", "payment_installments"]]

# Sentinela SIN_PAGO: cubre pedidos sin registro en olist_order_payments_dataset.
# Garantiza integridad referencial 100% en fact_ventas.
sin_pago = pd.DataFrame([{"pago_key": 0, "payment_type": "sin_pago", "payment_installments": 0}])
dim_pago = pd.concat([sin_pago, dim_pago], ignore_index=True)



### 7. Construcción de `base` para fact/sabana de ventas


In [ ]:
#Preparamos tabla pase para fact
# Contar items por pedido para distribuir pagos
items_count = (
    items.groupby("order_id", as_index=False)
    .agg(num_items=("order_item_id", "count"))
)

# Tabla base
base = items.merge(
    orders,
    on="order_id",
    how="left"
)

base = base.merge(
    customers,
    on="customer_id",
    how="left"
)

base = base.merge(
    products,
    on="product_id",
    how="left"
)

base = base.merge(
    sellers,
    on="seller_id",
    how="left"
)

base = base.merge(
    items_count,
    on="order_id",
    how="left"
)

base = base.merge(
    payments_agg,
    on="order_id",
    how="left"
)

# Pedidos sin registro de pago: mapear a la sentinela SIN_PAGO (pago_key=0).
base["pago_key"] = base["pago_key"].fillna(0).astype(int)
base["payment_type"] = base["payment_type"].fillna("sin_pago")
base["payment_installments"] = base["payment_installments"].fillna(0).astype(int)

base = base.merge(
    reviews_agg,
    on="order_id",
    how="left"
)

# Geolocalización cliente
base = base.merge(
    geolocation_agg.rename(columns={"zip_code_prefix": "customer_zip_code_prefix"}),
    on="customer_zip_code_prefix",
    how="left"
)

# Geolocalización vendedor
base = base.merge(
    geo_seller,
    on="seller_zip_code_prefix",
    how="left"
)

#Feature engineering en la tabla base
# Valor total del item
base["total_item_value"] = base["price"].fillna(0) + base["freight_value"].fillna(0)

# Pago asignado por item
base["payment_value_item"] = base["payment_value"] / base["num_items"]
base["payment_value_item"] = base["payment_value_item"].fillna(0)

# Indicadores de reseña
base["is_bad_review"] = (base["review_score"] <= 2).astype(int)
base["is_good_review"] = (base["review_score"] >= 4).astype(int)



### 8. `fact_ventas` y `tad_ventas`


In [ ]:
#Tabla de fact
fact_ventas = base[[
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "customer_id",
    "fecha_key",
    "pago_key",
    "order_status",
    "price",
    "freight_value",
    "total_item_value",
    "payment_value_item",
    "review_score",
    "delivery_days_real",
    "delivery_days_estimated",
    "delivery_delay_days",
    "is_late_delivery",
    "is_delivered",
    "is_bad_review",
    "is_good_review",
    "customer_zip_code_prefix"
]].copy()

#sabana de ventas
tad_ventas = base[[
    "order_id",
    "order_item_id",
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "customer_zip_code_prefix",
    "geo_lat",
    "geo_lng",
    "geo_city",
    "geo_state",
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "product_volume_cm3",
    "seller_id",
    "seller_city",
    "seller_state",
    "seller_zip_code_prefix",
    "seller_geo_lat",
    "seller_geo_lng",
    "payment_type",
    "payment_installments",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "anio",
    "mes",
    "dia",
    "trimestre",
    "dia_semana_num",
    "dia_semana_nombre",
    "es_fin_semana",
    "price",
    "freight_value",
    "total_item_value",
    "payment_value_item",
    "review_score",
    "delivery_days_real",
    "delivery_days_estimated",
    "delivery_delay_days",
    "is_late_delivery",
    "is_delivered",
    "is_bad_review",
    "is_good_review"
]].copy()



### 9. `base_pedidos`, `fact_pedidos` y `tad_pedidos`


In [ ]:
#Base de pedidos
base_pedidos = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

base_pedidos = base_pedidos.merge(
    items_count,
    on="order_id",
    how="left"
)

base_pedidos = base_pedidos.merge(
    payments_agg,
    on="order_id",
    how="left"
)

base_pedidos = base_pedidos.merge(
    reviews_agg,
    on="order_id",
    how="left"
)

base_pedidos = base_pedidos.merge(
    geolocation_agg.rename(columns={"zip_code_prefix": "customer_zip_code_prefix"}),
    on="customer_zip_code_prefix",
    how="left"
)


#Feature engineering para pedidos
base_pedidos["num_items"] = base_pedidos["num_items"].fillna(0).astype(int)
base_pedidos["tiene_items"] = (base_pedidos["num_items"] > 0).astype(int)

base_pedidos["is_bad_review"] = (base_pedidos["review_score"] <= 2).astype(int)
base_pedidos["is_good_review"] = (base_pedidos["review_score"] >= 4).astype(int)

#Fact pedidos
fact_pedidos = base_pedidos[[
    "order_id",
    "customer_id",
    "fecha_key",
    "pago_key",
    "order_status",
    "num_items",
    "tiene_items",
    "payment_value",
    "review_score",
    "delivery_days_real",
    "delivery_days_estimated",
    "delivery_delay_days",
    "is_late_delivery",
    "is_delivered",
    "is_bad_review",
    "is_good_review",
    "customer_zip_code_prefix"
]].copy()

#Sabana pedidos
tad_pedidos = base_pedidos[[
    "order_id",
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "customer_zip_code_prefix",
    "geo_lat",
    "geo_lng",
    "geo_city",
    "geo_state",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "fecha",
    "fecha_key",
    "anio",
    "mes",
    "dia",
    "trimestre",
    "dia_semana_num",
    "dia_semana_nombre",
    "es_fin_semana",
    "num_items",
    "tiene_items",
    "payment_type",
    "payment_installments",
    "payment_value",
    "review_score",
    "flag_short_message",
    "flag_answer_before_creation",
    "flag_review_id_duplicated",
    "flag_review_id_multi_order",
    "delivery_days_real",
    "delivery_days_estimated",
    "delivery_delay_days",
    "is_late_delivery",
    "is_delivered",
    "is_bad_review",
    "is_good_review"
]].copy()



### 10. Exportación a CSV


In [ ]:
# Exportamos resultados locales
# Dimensiones y hechos/cubos -> csv/cubos/
guardar_cubo(dim_tiempo, '02_dim_tiempo.csv')
guardar_cubo(dim_cliente, '02_dim_cliente.csv')
guardar_cubo(dim_geografia_cliente, '02_dim_geografia_cliente.csv')
guardar_cubo(dim_producto, '02_dim_producto.csv')
guardar_cubo(dim_vendedor, '02_dim_vendedor.csv')
guardar_cubo(dim_pago, '02_dim_pago.csv')
guardar_cubo(fact_ventas, '02_fact_ventas.csv')
guardar_cubo(fact_pedidos, '02_fact_pedidos.csv')

# Sabanas -> csv/
guardar_sabana(tad_ventas, '02_tad_ventas.csv')
guardar_sabana(tad_pedidos, '02_tad_pedidos.csv')

print('\nETL local finalizado correctamente.')
print('Archivos generados:')
for nombre, info in PIPELINE_AUDIT['generados'].items():
    print(f"- {nombre} ({info['tipo_salida']}): {info['filas']:,} filas | {len(info['columnas'])} columnas | {info['ruta']}")

In [ ]:
# Inspeccion rapida post-ETL de las tablas de hechos y sabanas.
print('--- fact_ventas: primeras filas ---')
print(fact_ventas.head())
print('\n--- tad_ventas: primeras filas ---')
print(tad_ventas.head())

print('\n--- fact_ventas: top 20 columnas con mas nulos ---')
print(fact_ventas.isnull().sum().sort_values(ascending=False).head(20))
print('\n--- tad_ventas: top 20 columnas con mas nulos ---')
print(tad_ventas.isnull().sum().sort_values(ascending=False).head(20))

print('\n--- order_id distintos ---')
print('fact_ventas:', fact_ventas['order_id'].nunique())
print('tad_ventas :', tad_ventas['order_id'].nunique())


#Validaciones


##General de formas

In [ ]:

print("dim_tiempo:", dim_tiempo.shape)
print("dim_cliente:", dim_cliente.shape)
print("dim_geografia_cliente:", dim_geografia_cliente.shape)
print("dim_producto:", dim_producto.shape)
print("dim_vendedor:", dim_vendedor.shape)
print("dim_pago:", dim_pago.shape)
print("fact_ventas:", fact_ventas.shape)
print("tad_ventas:", tad_ventas.shape)

##Unicidad de en dimensiones
Verificamos si las dimensiones si tiene una llave primaria, todos son true así que si lo tienen.

In [ ]:
print("dim_tiempo fecha_key única:", dim_tiempo["fecha_key"].is_unique)
print("dim_cliente customer_id único:", dim_cliente["customer_id"].is_unique)
print("dim_producto product_id único:", dim_producto["product_id"].is_unique)
print("dim_vendedor seller_id único:", dim_vendedor["seller_id"].is_unique)
print("dim_pago pago_key único:", dim_pago["pago_key"].is_unique)
print("dim_geografia_cliente zip_code_prefix único:", dim_geografia_cliente["zip_code_prefix"].is_unique)

##Granularidad de FACT_VENTAS
No debe de haber duplicados

In [ ]:
print("¿order_id + order_item_id es único en fact_ventas?")
print(not fact_ventas.duplicated(["order_id", "order_item_id"]).any())

print("Filas duplicadas exactas en fact_ventas:")
print(fact_ventas.duplicated().sum())

##Integridad referencial
Es donde comprobamos que todas las llaves de fact existan en las dimensiones. Esperamos de preferencia un resultado igual a uno o muy cercano a uno.

In [ ]:
print("customer_id de fact existe en dim_cliente:",
      fact_ventas["customer_id"].isin(dim_cliente["customer_id"]).mean())

print("product_id de fact existe en dim_producto:",
      fact_ventas["product_id"].isin(dim_producto["product_id"]).mean())

print("seller_id de fact existe en dim_vendedor:",
      fact_ventas["seller_id"].isin(dim_vendedor["seller_id"]).mean())

print("fecha_key de fact existe en dim_tiempo:",
      fact_ventas["fecha_key"].isin(dim_tiempo["fecha_key"]).mean())

print("pago_key de fact existe en dim_pago:",
      fact_ventas["pago_key"].isin(dim_pago["pago_key"]).mean())

print("zip de fact existe en dim_geografia_cliente:",
      fact_ventas["customer_zip_code_prefix"].isin(dim_geografia_cliente["zip_code_prefix"]).mean())

##Nulos críticos.


In [ ]:
# Nulos en columnas criticas de fact_ventas (deberian ser 0).
campos_criticos = [
    'order_id', 'order_item_id', 'customer_id', 'product_id', 'seller_id',
    'fecha_key', 'price', 'freight_value', 'total_item_value',
]
print('Nulos en campos criticos de fact_ventas:')
print(fact_ventas[campos_criticos].isnull().sum())


##Métricas

Verificamos que los montos no se hayan inflado.

In [ ]:
print("Suma total de price en items:")
print(items["price"].sum())

print("Suma total de price en fact_ventas:")
print(fact_ventas["price"].sum())

print("Suma total de freight_value en items:")
print(items["freight_value"].sum())

print("Suma total de freight_value en fact_ventas:")
print(fact_ventas["freight_value"].sum())

print("Suma total de total_item_value en fact_ventas:")
print(fact_ventas["total_item_value"].sum())

print("Suma total de payment_value en payments:")
print(payments["payment_value"].sum())

print("Suma total de payment_value_item en fact_ventas:")
print(fact_ventas["payment_value_item"].sum())


Notemos que

price en items = price en fact_ventas

freight_value en items = freight_value en fact_ventas

total_item_value = price + freight_value

Todos cuadran entonces eso nos garantiza la granularidad de fact.

Pero tenemos que la suma de pagos en la tabla de hechos no coincide con la suma total de la tabla original de pagos porque la tabla de hechos fue construida a nivel item de pedido, por lo que únicamente incorpora pedidos presentes en order_items. En consecuencia, pagos asociados a pedidos fuera de dicha intersección no forman parte del cubo analítico.


Para esto solo vamos a hacer la suma del total de pagos solo para pedidos que existen en items.

In [ ]:
# Total de pagos solo para pedidos que sí existen en items
payments_en_items = payments.loc[payments["order_id"].isin(items["order_id"])]

print("Suma total de payment_value en payments (solo pedidos con items):")
print(payments_en_items["payment_value"].sum())

print("Suma total de payment_value_item en fact_ventas:")
print(fact_ventas["payment_value_item"].sum())

Se validó la consistencia de las métricas monetarias entre las tablas fuente y la tabla de hechos. En particular, el valor de pago fue distribuido proporcionalmente por ítem de pedido para respetar la granularidad del cubo. La validación mostró que la suma de payment_value_item en la tabla de hechos coincide exactamente con la suma de pagos de los pedidos que cuentan con ítems asociados, confirmando que no existen duplicaciones por el proceso de integración.

##Cobertura necesaria para el dahsboard.


In [ ]:
print("tad_ventas:", tad_ventas.shape)
print("tad_pedidos:", tad_pedidos.shape)

print("\n¿order_id + order_item_id es único en tad_ventas?")
print(not tad_ventas.duplicated(["order_id", "order_item_id"]).any())

print("\n¿order_id es único en tad_pedidos?")
print(tad_pedidos["order_id"].is_unique)

print("\nNulos principales en tad_ventas:")
print(tad_ventas[[
    "order_id", "order_item_id", "customer_id", "product_id", "seller_id"
]].isnull().sum())

print("\nNulos principales en tad_pedidos:")
print(tad_pedidos[[
    "order_id", "customer_id", "order_status", "tiene_items"
]].isnull().sum())

print("\nDistribución de tiene_items en tad_pedidos:")
print(tad_pedidos["tiene_items"].value_counts(dropna=False))

print("\nTop order_status en tad_pedidos:")
print(tad_pedidos["order_status"].value_counts(dropna=False))

## Conclusión de validación

A partir de las pruebas realizadas se concluye que:

- las dimensiones cuentan con llaves únicas,
- la granularidad de la tabla de hechos se conserva correctamente,
- no existen inflaciones en las métricas monetarias,
- las relaciones entre hechos y dimensiones presentan alta integridad,
- los nulos críticos son mínimos o controlados,
- y las tablas planas tad_ventas y tad_pedidos son aptas para su consumo en dashboard.

Por lo tanto, la capa analítica se considera validada para su carga en BigQuery y posterior visualización en Looker Studio.

## Validacion local de salidas


In [ ]:
archivos_sabanas = ['02_tad_ventas.csv', '02_tad_pedidos.csv']
archivos_cubos = [
    '02_dim_tiempo.csv',
    '02_dim_cliente.csv',
    '02_dim_geografia_cliente.csv',
    '02_dim_producto.csv',
    '02_dim_vendedor.csv',
    '02_dim_pago.csv',
    '02_fact_ventas.csv',
    '02_fact_pedidos.csv',
]

print('=' * 80)
print('VALIDACION LOCAL - SCRIPT 02')
print('=' * 80)
print(f'Entrada limpios : {os.path.abspath(CSV_LIMPIOS_DIR)}')
print(f'Salida sabanas  : {os.path.abspath(SABANAS_DIR)}')
print(f'Salida cubos    : {os.path.abspath(CUBOS_DIR)}')
print(f'SUBIR_A_BIGQUERY: {SUBIR_A_BIGQUERY}')

print('\nArchivos leidos:')
for nombre, info in PIPELINE_AUDIT['leidos'].items():
    print(f"- {nombre}: {info['filas']:,} filas | {len(info['columnas'])} columnas | duplicados exactos: {info['duplicados_exactos']}")
    print(f"  Tipos principales: {info['tipos_principales']}")
    print(f"  Nulos principales: {info['nulos_principales']}")

print('\nArchivos generados:')
for nombre, info in PIPELINE_AUDIT['generados'].items():
    print(f"- {nombre} ({info['tipo_salida']}): {info['filas']:,} filas | {len(info['columnas'])} columnas | duplicados exactos: {info['duplicados_exactos']}")
    print(f"  Ruta: {info['ruta']}")
    print(f"  Columnas finales: {info['columnas']}")
    print(f"  Tipos principales: {info['tipos_principales']}")
    print(f"  Nulos principales: {info['nulos_principales']}")

faltan_sabanas = [nombre for nombre in archivos_sabanas if not os.path.exists(path_sabana(nombre))]
faltan_cubos = [nombre for nombre in archivos_cubos if not os.path.exists(path_cubo(nombre))]

assert not faltan_sabanas, f'Faltan sabanas: {faltan_sabanas}'
assert not faltan_cubos, f'Faltan cubos/dimensiones/hechos: {faltan_cubos}'
assert tad_ventas['product_category_name_english'].notna().all(), 'Quedan nulos en product_category_name_english'
assert not tad_ventas.duplicated(['order_id', 'order_item_id']).any(), 'tad_ventas no conserva granularidad order_id + order_item_id'
assert tad_pedidos['order_id'].is_unique, 'tad_pedidos no conserva granularidad por order_id'

print('\nOK: sabanas y cubos se generaron correctamente en local. BigQuery no se ejecuto.')

#Subir sabanas a bigquery


In [ ]:
# Dependencias BigQuery: desactivadas por defecto.
# Cambiar SUBIR_A_BIGQUERY = True solo cuando ya se quiera cargar a BigQuery.
if SUBIR_A_BIGQUERY:
    import subprocess
    import sys

    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'google-cloud-bigquery',
        'pandas-gbq',
        'pyarrow',
        'db-dtypes',
    ])
else:
    print('BigQuery desactivado: no se instalan dependencias ni se carga nada.')

In [ ]:
# Auth para BigQuery: usar Application Default Credentials.
# En local: `gcloud auth application-default login` o exportar
# GOOGLE_APPLICATION_CREDENTIALS apuntando al JSON de la service account.
if SUBIR_A_BIGQUERY:
    if not os.environ.get('GOOGLE_APPLICATION_CREDENTIALS') and not os.path.exists(os.path.expanduser('~/.config/gcloud/application_default_credentials.json')):
        print('Aviso: no se encontraron credenciales ADC. Configura GOOGLE_APPLICATION_CREDENTIALS o ejecuta `gcloud auth application-default login`.')
else:
    print('BigQuery desactivado: se omite validacion de credenciales ADC.')

In [ ]:
# Credencial del proyecto BigQuery: se espera el JSON en CWD o en la ruta de la variable.
if SUBIR_A_BIGQUERY:
    JSON_INFO_PATH = os.environ.get('SSC_CREDENTIALS', 'smart_supply_chain.json')
    assert os.path.exists(JSON_INFO_PATH), f'No se encontro {JSON_INFO_PATH}. Coloca el JSON de credenciales en el directorio o exporta SSC_CREDENTIALS.'
else:
    JSON_INFO_PATH = None
    print('BigQuery desactivado: no se lee JSON de credenciales.')

In [ ]:
if SUBIR_A_BIGQUERY:
    import json

    # JSON_INFO_PATH viene definido en la celda anterior (env SSC_CREDENTIALS o default).
    with open(JSON_INFO_PATH, 'r', encoding='utf-8') as f:
        cred_info = json.load(f)

    project_id = cred_info['proyecto']
    dataset_id = cred_info['dataset']

    print('Proyecto detectado:', project_id)
    print('Dataset detectado:', dataset_id)
else:
    project_id = None
    dataset_id = None
    print('BigQuery desactivado: no se detecta proyecto/dataset.')

In [ ]:
if SUBIR_A_BIGQUERY:
    from google.cloud import bigquery

    client = bigquery.Client(project=project_id)

    print('Cliente BigQuery creado correctamente.')
else:
    client = None
    print('BigQuery desactivado: no se crea cliente.')

In [ ]:
if SUBIR_A_BIGQUERY:
    datasets = [ds.dataset_id for ds in client.list_datasets()]
    print('Datasets visibles en el proyecto:')
    for ds in datasets:
        print('-', ds)

    if dataset_id not in datasets:
        raise ValueError(f"El dataset '{dataset_id}' no existe o no es visible con tu cuenta.")

    print(f"\nDataset '{dataset_id}' verificado correctamente.")
else:
    print('BigQuery desactivado: no se listan datasets.')

In [ ]:
print('tad_ventas:', tad_ventas.shape)
print('tad_pedidos:', tad_pedidos.shape)

print('\norder_id + order_item_id es unico en tad_ventas?')
print(not tad_ventas.duplicated(['order_id', 'order_item_id']).any())

print('\norder_id es unico en tad_pedidos?')
print(tad_pedidos['order_id'].is_unique)

In [ ]:
# Limpieza antes de entrar a BigQuery.
# Se preparan copias compatibles, pero solo se usaran si SUBIR_A_BIGQUERY = True.
tad_ventas_bq = tad_ventas.copy()
tad_pedidos_bq = tad_pedidos.copy()

for df in [tad_ventas_bq, tad_pedidos_bq]:
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            continue
        if df[col].dtype == 'object':
            df[col] = df[col].astype('string')

print('Copias para BigQuery preparadas correctamente.')
if not SUBIR_A_BIGQUERY:
    print('BigQuery desactivado: estas copias no se cargaran.')

##Función para pasar de datagrames a bigquery

In [ ]:
def subir_dataframe_a_bigquery(df, nombre_tabla, write_disposition='WRITE_TRUNCATE'):
    if not SUBIR_A_BIGQUERY:
        print(f"BigQuery desactivado: no se carga la tabla {nombre_tabla}.")
        return None

    table_id = f'{project_id}.{dataset_id}.{nombre_tabla}'

    job_config = bigquery.LoadJobConfig(
        write_disposition=write_disposition,
        autodetect=True
    )

    job = client.load_table_from_dataframe(
        df,
        table_id,
        job_config=job_config
    )

    job.result()

    tabla = client.get_table(table_id)
    print(f'\nTabla cargada: {table_id}')
    print(f'Filas: {tabla.num_rows}')
    print(f'Columnas: {len(tabla.schema)}')
    return tabla

In [ ]:
if SUBIR_A_BIGQUERY:
    subir_dataframe_a_bigquery(tad_pedidos_bq, 'tad_pedidos')
    subir_dataframe_a_bigquery(tad_ventas_bq, 'tad_ventas')
else:
    print('BigQuery desactivado: carga omitida. Para cargar, cambia SUBIR_A_BIGQUERY = True y configura credenciales fuera del repo.')